In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from head_tail import  extract_head_tail

In [3]:
def solve_book(input_csv, output_csv, model_name='all-MiniLM-L6-v2'):
    print(f"Loading data from {input_csv}...")
    df = pd.read_csv(input_csv)
    
    # Extract page identifiers and text
    page_ids = df['page'].tolist()
    texts = df['text'].tolist()
    
    print("Extracting Heads and Tails...")
    heads, tails = zip(*[extract_head_tail(t) for t in texts])
    
    print(f"Loading Sentence Transformer ({model_name})...")
    # Using a fast, open-source model. You can upgrade this to 'bge-large-en-v1.5' later
    model = SentenceTransformer(model_name)
    
    print("Computing Embeddings...")
    head_embeddings = model.encode(heads, show_progress_bar=True)
    tail_embeddings = model.encode(tails, show_progress_bar=True)
    
    print("Calculating Transition Matrix...")
    # S[i, j] represents how well the Tail of page i matches the Head of page j
    transition_matrix = cosine_similarity(tail_embeddings, head_embeddings)
    
    # We cannot transition from a page to itself, so set the diagonal to -infinity
    np.fill_diagonal(transition_matrix, -np.inf)
    
    print("Reconstructing Narrative (Greedy Chain)...")
    num_pages = len(page_ids)
    
    # Heuristic: Find the starting page. 
    # A starting page is likely to have very low incoming similarity scores.
    max_incoming_scores = np.max(transition_matrix, axis=0)
    current_page_idx = np.argmin(max_incoming_scores)
    
    ordered_indices = [current_page_idx]
    visited = set([current_page_idx])
    
    # Greedily pick the next most likely page
    for _ in range(num_pages - 1):
        # Get transition scores from the current page
        scores_from_current = transition_matrix[current_page_idx].copy()
        
        # Ignore pages we've already visited
        for v in visited:
            scores_from_current[v] = -np.inf
            
        # The next page is the unvisited page with the highest similarity score
        next_page_idx = np.argmax(scores_from_current)
        
        ordered_indices.append(next_page_idx)
        visited.add(next_page_idx)
        current_page_idx = next_page_idx

    print("Formatting Submission...")
    # Create the submission dataframe
    # The evaluation requires 1-based indexing for original_page
    submission_data = {
        'original_page': range(1, num_pages + 1),
        'shuffled_page': [page_ids[i] for i in ordered_indices]
    }
    
    sub_df = pd.DataFrame(submission_data)
    sub_df.to_csv(output_csv, index=False)
    print(f"Saved reconstruction to {output_csv}\n")

In [4]:
try:
    solve_book('./data/BookA_test.csv', './submissions/BookA_bi_encoder.csv', model_name='./models/fine_tuned_bi_encoder')
    solve_book('./data/BookB_test.csv', './submissions/BookB_bi_encoder.csv', model_name='./models/fine_tuned_bi_encoder')
    print("Success! You can now submit these CSV files.")
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure your dataset files are named correctly and in the right folder.")

Loading data from ./data/BookA_test.csv...
Extracting Heads and Tails...
Loading Sentence Transformer (./models/fine_tuned_bi_encoder)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4477.89it/s]


Computing Embeddings...


Batches: 100%|██████████| 5/5 [00:03<00:00,  1.42it/s]


Calculating Transition Matrix...
Reconstructing Narrative (Greedy Chain)...
Formatting Submission...
Saved reconstruction to ./submissions/BookA_bi_encoder.csv

Loading data from ./data/BookB_test.csv...
Extracting Heads and Tails...
Loading Sentence Transformer (./models/fine_tuned_bi_encoder)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2123.28it/s]


Computing Embeddings...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.56it/s]

Calculating Transition Matrix...
Reconstructing Narrative (Greedy Chain)...
Formatting Submission...
Saved reconstruction to ./submissions/BookB_bi_encoder.csv

Success! You can now submit these CSV files.
